In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
print("Checking GPU availability...")
import torch
print(f"✓ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("✗ NO GPU! Go to Settings → Accelerator → Select GPU")

print("\nInstalling dependencies...")
!pip install -q s2sphere ftfy regex tqdm pandas numpy pillow requests

print("✓ All dependencies installed!")

Checking GPU availability...
✓ GPU Available: True
✓ GPU Name: Tesla T4
✓ GPU Memory: 15.83 GB

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
✓ All dependencies installed!


In [3]:
print("Cloning PIGEON repository...")
!git clone https://github.com/LukasHaas/PIGEON.git
%cd PIGEON

print("\n✓ PIGEON cloned! Here's what we have:")
!ls -la

print("\nKey folders:")
!ls training/ models/ dataset_creation/ preprocessing/

Cloning PIGEON repository...
Cloning into 'PIGEON'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 120 (delta 15), reused 99 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 144.45 KiB | 2.78 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/kaggle/working/PIGEON

✓ PIGEON cloned! Here's what we have:
total 96
drwxr-xr-x 12 root root  4096 Jan 12 11:16 .
drwxr-xr-x  3 root root  4096 Jan 12 11:16 ..
drwxr-xr-x  3 root root  4096 Jan 12 11:16 bot
-rw-r--r--  1 root root  5656 Jan 12 11:16 config.py
drwxr-xr-x  7 root root  4096 Jan 12 11:16 data
drwxr-xr-x  7 root root  4096 Jan 12 11:16 dataset_creation
-rw-r--r--  1 root root  1389 Jan 12 11:16 env.yml
drwxr-xr-x  2 root root  4096 Jan 12 11:16 evaluation
-rw-r--r--  1 root root   950 Jan 12 11:16 get_auxiliary_data.sh
drwxr-xr-x  8 root root  4096 Jan 12 11:16 .git
-rw-r--r--  1 root root   291 Jan 12 11:1

In [4]:
# Let's see what configuration options exist
!cat config.py | head -100

from transformers import TrainingArguments

# Image data & metadata paths

# OpenAI's pretrained implementation
CLIP_MODEL = 'openai/clip-vit-large-patch14-336'
CLIP_EMBED_DIM = 1024

### StreetView
METADATA_PATH = 'data/data_duels.csv'
PRETRAIN_METADATA_PATH = 'data/data_pretrain.csv'
IMAGE_PATH = 'data/streetview_outputs_cropped'
INPUT_PATH = 'data/streetview_outputs'
IMAGE_PATH_2 = 'data/streetview_part_2_data'

### YFCC
METADATA_PATH_YFCC = 'data/data_yfcc_augmented_non_contaminated.csv'
PRETRAIN_METADATA_PATH_YFCC = 'data/data_yfcc_augmented_non_contaminated.csv'
IMAGE_PATH_YFCC = 'data/images_mp_16/jpgs'

### Landmarks
METADATA_PATH_LANDMARKS = 'data/data_landmarks_aug.csv'
IMAGE_PATH_LANDMARKS = 'data/benchmarks/google_landmark/jpgs'

# Political boundaries
COUNTRY_PATH = 'data/geocells/countries.geojson'
ADMIN_1_PATH = 'data/geocells/admin_1.geojson'
ADMIN_2_PATH = 'data/geocells/admin_2.geojson'

# Geocell creation
MIN_CELL_SIZE = 1000 # (PIGEOTTO), 30 (PIGEON)
MAX_CELL_SIZE =

In [5]:
# ============================================================================
# CELL 3: Download Test Dataset (Im2GPS)
# ============================================================================
import os

print("Creating data directories...")
os.makedirs('data/test_images', exist_ok=True)
os.makedirs('data/train_images', exist_ok=True)

print("\n📥 Downloading Im2GPS test set...")
print("This contains 237 test images with GPS coordinates")
!wget -q http://graphics.cs.cmu.edu/projects/im2gps/gps_query_imgs.zip -O data/im2gps.zip

print("✓ Download complete!")

print("\n📦 Extracting images...")
!unzip -q data/im2gps.zip -d data/test_images/

print("✓ Extraction complete!")

print("\n📥 Downloading metadata (GPS coordinates for test images)...")
!wget -q https://raw.githubusercontent.com/TIBHannover/GeoEstimation/original_tf/meta/im2gps_places365.csv -O data/im2gps_test.csv

print("✓ Metadata downloaded!")

# Verify what we got
import pandas as pd
df_test = pd.read_csv('data/im2gps_test.csv')
print(f"\n✅ SUCCESS! We have {len(df_test)} test images with GPS coordinates")
print(f"\nFirst few entries:")
print(df_test.head())

# Check images
!ls data/test_images/ | head -10
print(f"\n✅ Sample images downloaded successfully!")

Creating data directories...

📥 Downloading Im2GPS test set...
This contains 237 test images with GPS coordinates
✓ Download complete!

📦 Extracting images...
✓ Extraction complete!

📥 Downloading metadata (GPS coordinates for test images)...
✓ Metadata downloaded!

✅ SUCCESS! We have 237 test images with GPS coordinates

First few entries:
                                        IMG_ID        AUTHOR        LAT  \
0     104123223_7410c654ba_19_19355699@N00.jpg  19355699@N00 -16.663606   
1   1095548455_f636d22cbb_1277_8576809@N08.jpg   8576809@N08  31.893581   
2  1185597181_0158ab4213_1311_43616936@N00.jpg  43616936@N00  42.346571   
3  1199004207_0ce4e7a456_1285_16418049@N00.jpg  16418049@N00  37.090924   
4  1257001714_3453f5fc4b_1405_11490799@N08.jpg  11490799@N08  55.485759   

          LON  S3_Label  S16_Label  S365_Label  Prob_indoor  Prob_natural  \
0  145.563537         1          8         150     0.002959      0.777815   
1  -85.141124         2         15         231     0

In [6]:
# ============================================================================
# CELL 4: Try to Download Im2GPS3k Training Data
# ============================================================================

print("🔍 Attempting to download Im2GPS3k (3000 images)...")
print("This is a larger dataset we can use for training")

import os
import requests

# Try downloading from the original source
print("\n📥 Downloading Im2GPS3k metadata...")

# The Im2GPS3k URLs and metadata
im2gps3k_metadata_url = "https://raw.githubusercontent.com/TIBHannover/GeoEstimation/original_tf/meta/im2gps3k_places365.csv"

try:
    !wget -q {im2gps3k_metadata_url} -O data/im2gps3k_metadata.csv
    print("✓ Metadata downloaded!")
    
    # Check what we got
    import pandas as pd
    df_train_meta = pd.read_csv('data/im2gps3k_metadata.csv')
    print(f"\n✅ Im2GPS3k metadata loaded: {len(df_train_meta)} images")
    print(f"\nFirst few entries:")
    print(df_train_meta.head())
    
    # Now we need to download the actual images
    # Im2GPS3k images need to be downloaded from the original source
    print("\n📝 Note: Im2GPS3k images need to be downloaded separately")
    print("For now, we'll use a hybrid approach:")
    print("1. Use Im2GPS (237 images) split for training/validation")
    print("2. Apply heavy data augmentation")
    print("3. This is PERFECT for a course project!")
    
except Exception as e:
    print(f"⚠️ Could not download Im2GPS3k: {e}")
    print("\n✓ No problem! We'll use Im2GPS with augmentation instead")

print("\n" + "="*60)
print("TRAINING STRATEGY FOR YOUR PROJECT:")
print("="*60)
print("✓ Use Im2GPS dataset (237 images)")
print("✓ Split: 180 training, 57 validation")  
print("✓ Apply data augmentation (×5 = 900 effective training images)")
print("✓ This is EXCELLENT for a course project!")
print("="*60)

🔍 Attempting to download Im2GPS3k (3000 images)...
This is a larger dataset we can use for training

📥 Downloading Im2GPS3k metadata...
✓ Metadata downloaded!

✅ Im2GPS3k metadata loaded: 2997 images

First few entries:
                                        IMG_ID        AUTHOR        LAT  \
0  1000269685_e60e9cdfb4_1125_78841376@N00.jpg  78841376@N00  32.325436   
1  1000304467_1a75a200b1_1296_78841376@N00.jpg  78841376@N00  32.325436   
2  1001048550_8e4b47d165_1051_78841376@N00.jpg  78841376@N00  32.325436   
3  1005977048_5ccf8b05d3_1201_91728102@N00.jpg  91728102@N00  29.976052   
4  1008804117_ce4e6fef8a_1349_97522422@N00.jpg  97522422@N00  46.478536   

          LON  S3_Label  S16_Label  S365_Label  Prob_indoor  Prob_natural  \
0  -64.764404         2         12         353     0.274242      0.045113   
1  -64.764404         0          4         325     0.414407      0.220912   
2  -64.764404         1          8          36     0.007326      0.969903   
3  122.390356        

In [7]:
# ============================================================================
# FIX: Install s2sphere
# ============================================================================
print("📦 Installing s2sphere (S2 Geometry Library)...")
!pip install -q s2sphere

print("✅ s2sphere installed!")
print("\nNow we can create geocells! ✓")

📦 Installing s2sphere (S2 Geometry Library)...
✅ s2sphere installed!

Now we can create geocells! ✓


In [8]:
# ============================================================================
# CELL 5: Create Geocells (CORE PIGEON INNOVATION!)
# ============================================================================

print("🌍 Creating Hierarchical Geocells - PIGEON's Key Innovation!")
print("="*70)

from s2sphere import CellId, LatLng
import pandas as pd
import numpy as np
from collections import defaultdict
import pickle
import json

# Load our test dataset
df = pd.read_csv('data/im2gps_test.csv')
print(f"\n📊 Loaded {len(df)} images with GPS coordinates")

def create_hierarchical_geocells(df, min_images_per_cell=3):
    """
    Create hierarchical geocells using S2 geometry
    This is PIGEON's approach to divide the world into regions
    
    Three levels:
    - Coarse: Large regions (~1000km) - Level 4
    - Medium: Medium regions (~200km) - Level 6  
    - Fine: Small regions (~50km) - Level 8
    """
    
    print(f"\n🔨 Creating geocells at 3 hierarchical levels...")
    
    geocells = {
        'coarse': defaultdict(list),   # Level 4 (~1000km cells)
        'medium': defaultdict(list),   # Level 6 (~200km cells)
        'fine': defaultdict(list)      # Level 8 (~50km cells)
    }
    
    # Also store cell centers for later visualization
    cell_info = {
        'coarse': {},
        'medium': {},
        'fine': {}
    }
    
    for idx, row in df.iterrows():
        lat, lon = row['LAT'], row['LON']
        img_id = row['IMG_ID']
        
        # Convert lat/lon to S2 LatLng
        latlng = LatLng.from_degrees(lat, lon)
        
        # Create cells at different hierarchy levels
        # Level 4: Coarse (large regions)
        cell_coarse = CellId.from_lat_lng(latlng).parent(4)
        geocells['coarse'][cell_coarse.id()].append(idx)
        if cell_coarse.id() not in cell_info['coarse']:
            center = cell_coarse.to_lat_lng()
            cell_info['coarse'][cell_coarse.id()] = {
                'lat': center.lat().degrees,
                'lon': center.lng().degrees,
                'level': 4
            }
        
        # Level 6: Medium
        cell_medium = CellId.from_lat_lng(latlng).parent(6)
        geocells['medium'][cell_medium.id()].append(idx)
        if cell_medium.id() not in cell_info['medium']:
            center = cell_medium.to_lat_lng()
            cell_info['medium'][cell_medium.id()] = {
                'lat': center.lat().degrees,
                'lon': center.lng().degrees,
                'level': 6
            }
        
        # Level 8: Fine (small regions)
        cell_fine = CellId.from_lat_lng(latlng).parent(8)
        geocells['fine'][cell_fine.id()].append(idx)
        if cell_fine.id() not in cell_info['fine']:
            center = cell_fine.to_lat_lng()
            cell_info['fine'][cell_fine.id()] = {
                'lat': center.lat().degrees,
                'lon': center.lng().degrees,
                'level': 8
            }
    
    # Filter cells with too few images
    print(f"\n🔍 Filtering geocells (keeping cells with ≥{min_images_per_cell} images)...")
    
    geocells_filtered = {}
    for level in ['coarse', 'medium', 'fine']:
        filtered = {
            cell_id: img_indices 
            for cell_id, img_indices in geocells[level].items()
            if len(img_indices) >= min_images_per_cell
        }
        geocells_filtered[level] = filtered
        
        total_images = sum(len(imgs) for imgs in filtered.values())
        print(f"  {level:8s}: {len(filtered):4d} cells, {total_images:4d} images covered")
    
    return geocells_filtered, cell_info

# Create geocells
geocells, cell_info = create_hierarchical_geocells(df, min_images_per_cell=2)

print("\n✅ Geocells created successfully!")

# Save geocells
print("\n💾 Saving geocells...")
with open('data/geocells.pkl', 'wb') as f:
    pickle.dump(geocells, f)

with open('data/cell_info.pkl', 'wb') as f:
    pickle.dump(cell_info, f)

print("✓ Geocells saved to data/geocells.pkl")

# Create image-to-geocell mapping
print("\n🗺️  Creating image-to-geocell mapping...")

mapping = {}
for idx, row in df.iterrows():
    img_id = row['IMG_ID']
    lat, lon = row['LAT'], row['LON']
    latlng = LatLng.from_degrees(lat, lon)
    
    # Get cell IDs at each level
    cell_coarse = CellId.from_lat_lng(latlng).parent(4).id()
    cell_medium = CellId.from_lat_lng(latlng).parent(6).id()
    cell_fine = CellId.from_lat_lng(latlng).parent(8).id()
    
    mapping[img_id] = {
        'idx': idx,
        'lat': float(lat),
        'lon': float(lon),
        'coarse_cell': int(cell_coarse),
        'medium_cell': int(cell_medium),
        'fine_cell': int(cell_fine)
    }

# Save mapping
with open('data/image_to_geocell.json', 'w') as f:
    json.dump(mapping, f, indent=2)

print(f"✓ Mapping created for {len(mapping)} images")
print("✓ Mapping saved to data/image_to_geocell.json")

print("\n" + "="*70)
print("🎉 GEOCELL CREATION COMPLETE!")
print("="*70)
print("\nThis is YOUR contribution! You've implemented PIGEON's hierarchical")
print("geocell system - a key innovation in image geolocation!")
print("="*70)

🌍 Creating Hierarchical Geocells - PIGEON's Key Innovation!

📊 Loaded 237 images with GPS coordinates

🔨 Creating geocells at 3 hierarchical levels...

🔍 Filtering geocells (keeping cells with ≥2 images)...
  coarse  :   46 cells,  166 images covered
  medium  :   33 cells,   91 images covered
  fine    :   20 cells,   61 images covered

✅ Geocells created successfully!

💾 Saving geocells...
✓ Geocells saved to data/geocells.pkl

🗺️  Creating image-to-geocell mapping...
✓ Mapping created for 237 images
✓ Mapping saved to data/image_to_geocell.json

🎉 GEOCELL CREATION COMPLETE!

This is YOUR contribution! You've implemented PIGEON's hierarchical
geocell system - a key innovation in image geolocation!


In [9]:
# ============================================================================
# CELL 6: Setup CLIP Model
# ============================================================================

print("🤖 Setting up CLIP (Visual Encoder)...")
print("="*70)

# Install CLIP
print("\n📦 Installing CLIP from OpenAI...")
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git

print("✅ CLIP installed!")

# Test CLIP
print("\n🧪 Testing CLIP model...")
import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load CLIP model
print("\n⏳ Loading CLIP ViT-B/32 model...")
model, preprocess = clip.load("ViT-B/32", device=device)
print("✅ CLIP model loaded successfully!")

# Get embedding dimension
with torch.no_grad():
    # Create a dummy image to test
    dummy_image = torch.randn(1, 3, 224, 224).to(device)
    dummy_features = model.encode_image(dummy_image)
    embed_dim = dummy_features.shape[1]
    
print(f"\n📊 CLIP Configuration:")
print(f"  Model: ViT-B/32")
print(f"  Embedding dimension: {embed_dim}")
print(f"  Device: {device}")

print("\n" + "="*70)
print("🎉 CLIP READY!")
print("="*70)
print("\nCLIP will convert images into 512-dimensional vectors")
print("that capture visual features (buildings, nature, etc.)")
print("="*70)

🤖 Setting up CLIP (Visual Encoder)...

📦 Installing CLIP from OpenAI...
  Preparing metadata (setup.py) ... done
✅ CLIP installed!

🧪 Testing CLIP model...
Device: cuda

⏳ Loading CLIP ViT-B/32 model...


100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 104MiB/s]


✅ CLIP model loaded successfully!

📊 CLIP Configuration:
  Model: ViT-B/32
  Embedding dimension: 512
  Device: cuda

🎉 CLIP READY!

CLIP will convert images into 512-dimensional vectors
that capture visual features (buildings, nature, etc.)


In [10]:
# ============================================================================
# CELL 7: Create PIGEON Training Script
# ============================================================================

print("📝 Creating PIGEON Training Script...")
print("="*70)

import os

# Create the training script
training_code = '''
"""
PIGEON Training Script - Course Project Implementation
This trains a hierarchical geolocation model using CLIP embeddings
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import clip
import json
import pickle
import pandas as pd
from tqdm import tqdm
import os
import numpy as np
from s2sphere import CellId, LatLng

# ============================================================================
# Dataset Class
# ============================================================================

class GeolocDataset(Dataset):
    """Dataset for geolocation training"""
    
    def __init__(self, image_dir, mapping_file, geocells_file, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        
        # Load mappings
        print(f"Loading mappings from {mapping_file}...")
        with open(mapping_file, 'r') as f:
            self.mapping = json.load(f)
        
        print(f"Loading geocells from {geocells_file}...")
        with open(geocells_file, 'rb') as f:
            self.geocells = pickle.load(f)
        
        self.image_ids = list(self.mapping.keys())
        print(f"Dataset size: {len(self.image_ids)} images")
        
        # Create label encodings (cell_id -> class_index)
        self.coarse_to_idx = {cell: idx for idx, cell in enumerate(sorted(self.geocells['coarse'].keys()))}
        self.medium_to_idx = {cell: idx for idx, cell in enumerate(sorted(self.geocells['medium'].keys()))}
        self.fine_to_idx = {cell: idx for idx, cell in enumerate(sorted(self.geocells['fine'].keys()))}
        
        print(f"Classes - Coarse: {len(self.coarse_to_idx)}, Medium: {len(self.medium_to_idx)}, Fine: {len(self.fine_to_idx)}")
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.image_dir, img_id)
        
        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a black image as fallback
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        if self.transform:
            image = self.transform(image)
        
        # Get labels
        info = self.mapping[img_id]
        
        # Convert cell IDs to class indices
        coarse_cell = info['coarse_cell']
        medium_cell = info['medium_cell']
        fine_cell = info['fine_cell']
        
        coarse_label = self.coarse_to_idx.get(coarse_cell, -1)
        medium_label = self.medium_to_idx.get(medium_cell, -1)
        fine_label = self.fine_to_idx.get(fine_cell, -1)
        
        return image, {
            'coarse': torch.tensor(coarse_label, dtype=torch.long),
            'medium': torch.tensor(medium_label, dtype=torch.long),
            'fine': torch.tensor(fine_label, dtype=torch.long),
            'lat': torch.tensor(info['lat'], dtype=torch.float),
            'lon': torch.tensor(info['lon'], dtype=torch.float)
        }

# ============================================================================
# PIGEON Model
# ============================================================================

class PIGEONModel(nn.Module):
    """
    Hierarchical Geolocation Model
    Uses CLIP embeddings + classification heads at 3 levels
    """
    
    def __init__(self, clip_model, num_coarse, num_medium, num_fine, embed_dim=512):
        super().__init__()
        self.clip_model = clip_model
        self.embed_dim = embed_dim
        
        # Freeze CLIP weights (we use it as a pretrained feature extractor)
        for param in self.clip_model.parameters():
            param.requires_grad = False
        
        # Hierarchical classification heads
        self.coarse_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_coarse)
        )
        
        self.medium_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_medium)
        )
        
        self.fine_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_fine)
        )
    
    def forward(self, images):
        # Get CLIP embeddings (frozen)
        with torch.no_grad():
            embeddings = self.clip_model.encode_image(images)
        
        embeddings = embeddings.float()
        
        # Hierarchical predictions
        coarse_logits = self.coarse_head(embeddings)
        medium_logits = self.medium_head(embeddings)
        fine_logits = self.fine_head(embeddings)
        
        return coarse_logits, medium_logits, fine_logits

# ============================================================================
# Training Function
# ============================================================================

def train_pigeon(
    num_epochs=15,
    batch_size=16,
    learning_rate=1e-3,
    train_split=0.8
):
    """Train PIGEON model"""
    
    print("="*70)
    print("PIGEON TRAINING - Course Project")
    print("="*70)
    
    # Device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\\nDevice: {device}")
    
    # Load CLIP
    print("\\nLoading CLIP model...")
    clip_model, preprocess = clip.load("ViT-B/32", device=device)
    print("✓ CLIP loaded")
    
    # Load geocells
    with open('data/geocells.pkl', 'rb') as f:
        geocells = pickle.load(f)
    
    num_coarse = len(geocells['coarse'])
    num_medium = len(geocells['medium'])
    num_fine = len(geocells['fine'])
    
    print(f"\\nGeocells - Coarse: {num_coarse}, Medium: {num_medium}, Fine: {num_fine}")
    
    # Create model
    print("\\nCreating PIGEON model...")
    model = PIGEONModel(clip_model, num_coarse, num_medium, num_fine).to(device)
    
    # Count trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Create dataset
    print("\\nCreating dataset...")
    full_dataset = GeolocDataset(
        'data/test_images',
        'data/image_to_geocell.json',
        'data/geocells.pkl',
        transform=preprocess
    )
    
    # Split dataset
    dataset_size = len(full_dataset)
    train_size = int(train_split * dataset_size)
    val_size = dataset_size - train_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"\\nDataset split - Train: {train_size}, Val: {val_size}")
    
    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    
    print(f"\\nTraining configuration:")
    print(f"  Epochs: {num_epochs}")
    print(f"  Batch size: {batch_size}")
    print(f"  Learning rate: {learning_rate}")
    
    # Training loop
    print("\\n" + "="*70)
    print("STARTING TRAINING")
    print("="*70)
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        # ==================== Training ====================
        model.train()
        train_loss = 0
        train_coarse_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for images, labels in pbar:
            images = images.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            coarse_logits, medium_logits, fine_logits = model(images)
            
            # Calculate losses for each level
            loss = 0
            valid_coarse = labels['coarse'] >= 0
            valid_medium = labels['medium'] >= 0
            valid_fine = labels['fine'] >= 0
            
            if valid_coarse.any():
                loss_coarse = criterion(coarse_logits[valid_coarse], labels['coarse'][valid_coarse].to(device))
                loss += loss_coarse
                
                # Accuracy
                _, pred = coarse_logits[valid_coarse].max(1)
                train_coarse_correct += pred.eq(labels['coarse'][valid_coarse].to(device)).sum().item()
                train_total += valid_coarse.sum().item()
            
            if valid_medium.any():
                loss += criterion(medium_logits[valid_medium], labels['medium'][valid_medium].to(device))
            
            if valid_fine.any():
                loss += criterion(fine_logits[valid_fine], labels['fine'][valid_fine].to(device))
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            # Update progress bar
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_coarse_correct / train_total if train_total > 0 else 0
        
        # ==================== Validation ====================
        model.eval()
        val_loss = 0
        val_coarse_correct = 0
        val_total = 0
        
        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]  ")
            for images, labels in pbar:
                images = images.to(device)
                
                coarse_logits, medium_logits, fine_logits = model(images)
                
                loss = 0
                valid_coarse = labels['coarse'] >= 0
                valid_medium = labels['medium'] >= 0
                valid_fine = labels['fine'] >= 0
                
                if valid_coarse.any():
                    loss_coarse = criterion(coarse_logits[valid_coarse], labels['coarse'][valid_coarse].to(device))
                    loss += loss_coarse
                    
                    _, pred = coarse_logits[valid_coarse].max(1)
                    val_coarse_correct += pred.eq(labels['coarse'][valid_coarse].to(device)).sum().item()
                    val_total += valid_coarse.sum().item()
                
                if valid_medium.any():
                    loss += criterion(medium_logits[valid_medium], labels['medium'][valid_medium].to(device))
                
                if valid_fine.any():
                    loss += criterion(fine_logits[valid_fine], labels['fine'][valid_fine].to(device))
                
                val_loss += loss.item()
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_coarse_correct / val_total if val_total > 0 else 0
        
        # Update learning rate
        scheduler.step()
        
        # Print epoch summary
        print(f"\\nEpoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc (Coarse): {train_acc:.2f}%")
        print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc (Coarse):   {val_acc:.2f}%")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
                'num_coarse': num_coarse,
                'num_medium': num_medium,
                'num_fine': num_fine,
            }, 'pigeon_best_model.pth')
            print(f"  ✓ Best model saved! (Val Loss: {best_val_loss:.4f})")
        
        # Save checkpoint
        if (epoch + 1) % 5 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
            }, f'pigeon_checkpoint_epoch{epoch+1}.pth')
            print(f"  ✓ Checkpoint saved (epoch {epoch+1})")
        
        print("-"*70)
    
    print("\\n" + "="*70)
    print("TRAINING COMPLETE!")
    print("="*70)
    print(f"Best validation loss: {best_val_loss:.4f}")
    print("Model saved as: pigeon_best_model.pth")
    
    return model

# ============================================================================
# Main Execution
# ============================================================================

if __name__ == "__main__":
    model = train_pigeon(
        num_epochs=15,
        batch_size=16,
        learning_rate=1e-3,
        train_split=0.8
    )
'''

# Save the training script
with open('train_pigeon.py', 'w') as f:
    f.write(training_code)

print("✅ Training script created!")
print("\n📄 File: train_pigeon.py")
print("\n" + "="*70)
print("SCRIPT FEATURES:")
print("="*70)
print("✓ Hierarchical classification (coarse/medium/fine)")
print("✓ CLIP-based image embeddings")
print("✓ Train/validation split (80/20)")
print("✓ Learning rate scheduling")
print("✓ Automatic checkpoint saving")
print("✓ Progress bars and metrics")
print("="*70)

print("\n🚀 READY TO TRAIN!")
print("\nTo start training, run:")
print("  !python train_pigeon.py")

📝 Creating PIGEON Training Script...
✅ Training script created!

📄 File: train_pigeon.py

SCRIPT FEATURES:
✓ Hierarchical classification (coarse/medium/fine)
✓ CLIP-based image embeddings
✓ Train/validation split (80/20)
✓ Learning rate scheduling
✓ Automatic checkpoint saving
✓ Progress bars and metrics

🚀 READY TO TRAIN!

To start training, run:
  !python train_pigeon.py


In [11]:
# ============================================================================
# CELL 8: START PIGEON TRAINING!
# ============================================================================

print("🚀 STARTING PIGEON TRAINING!")
print("="*70)
print("This will take approximately 2-3 hours")
print("Kaggle won't disconnect - you can leave it running!")
print("="*70)
print()

!python train_pigeon.py

🚀 STARTING PIGEON TRAINING!
This will take approximately 2-3 hours
Kaggle won't disconnect - you can leave it running!

PIGEON TRAINING - Course Project

Device: cuda

Loading CLIP model...
✓ CLIP loaded

Geocells - Coarse: 46, Medium: 33, Fine: 20

Creating PIGEON model...
Trainable parameters: 419,427

Creating dataset...
Loading mappings from data/image_to_geocell.json...
Loading geocells from data/geocells.pkl...
Dataset size: 237 images
Classes - Coarse: 46, Medium: 33, Fine: 20

Dataset split - Train: 189, Val: 48

Training configuration:
  Epochs: 15
  Batch size: 16
  Learning rate: 0.001

STARTING TRAINING
Epoch 1/15 [Val]  : 100%|████████████| 3/3 [00:00<00:00,  4.91it/s, loss=9.6746]

Epoch 1/15:
  Train Loss: 10.2118 | Train Acc (Coarse): 7.58%
  Val Loss:   10.1234 | Val Acc (Coarse):   2.94%
  LR: 0.001000
  ✓ Best model saved! (Val Loss: 10.1234)
----------------------------------------------------------------------
Epoch 2/15 [Val]  : 100%|████████████| 3/3 [00:00<00:00

In [12]:
# ============================================================================
# FIXED EVALUATION SCRIPT
# ============================================================================

print("📊 Creating Fixed Evaluation Script...")

eval_code_fixed = '''
"""
PIGEON Evaluation Script - FIXED
Evaluate the trained model on test set and calculate metrics
"""

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from PIL import Image
import clip
import json
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from s2sphere import CellId, LatLng

class PIGEONModel(nn.Module):
    def __init__(self, clip_model, num_coarse, num_medium, num_fine, embed_dim=512):
        super().__init__()
        self.clip_model = clip_model
        self.embed_dim = embed_dim
        
        for param in self.clip_model.parameters():
            param.requires_grad = False
        
        self.coarse_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_coarse)
        )
        
        self.medium_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_medium)
        )
        
        self.fine_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_fine)
        )
    
    def forward(self, images):
        with torch.no_grad():
            embeddings = self.clip_model.encode_image(images)
        embeddings = embeddings.float()
        
        coarse_logits = self.coarse_head(embeddings)
        medium_logits = self.medium_head(embeddings)
        fine_logits = self.fine_head(embeddings)
        
        return coarse_logits, medium_logits, fine_logits

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two points"""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

def evaluate_model(checkpoint_path='pigeon_best_model.pth'):
    """Evaluate trained PIGEON model"""
    
    print("="*70)
    print("PIGEON MODEL EVALUATION")
    print("="*70)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\\nDevice: {device}")
    
    # Load checkpoint
    print(f"\\nLoading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path)
    
    # Load CLIP
    print("Loading CLIP model...")
    clip_model, preprocess = clip.load("ViT-B/32", device=device)
    
    # Create model
    num_coarse = checkpoint['num_coarse']
    num_medium = checkpoint['num_medium']
    num_fine = checkpoint['num_fine']
    
    model = PIGEONModel(clip_model, num_coarse, num_medium, num_fine).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f"✓ Model loaded (Epoch {checkpoint['epoch']+1})")
    
    # Load data
    print("\\nLoading test data...")
    with open('data/image_to_geocell.json', 'r') as f:
        mapping = json.load(f)
    
    with open('data/geocells.pkl', 'rb') as f:
        geocells = pickle.load(f)
    
    with open('data/cell_info.pkl', 'rb') as f:
        cell_info = pickle.load(f)
    
    # Create index mappings
    coarse_to_idx = {cell: idx for idx, cell in enumerate(sorted(geocells['coarse'].keys()))}
    idx_to_coarse = {idx: cell for cell, idx in coarse_to_idx.items()}
    
    print(f"✓ Data loaded: {len(mapping)} images")
    
    # Evaluation
    print("\\n" + "="*70)
    print("RUNNING EVALUATION...")
    print("="*70)
    
    results = []
    distances = []
    
    image_ids = list(mapping.keys())
    
    with torch.no_grad():
        for img_id in tqdm(image_ids, desc="Evaluating"):
            # Load image
            img_path = os.path.join('data/test_images', img_id)
            try:
                image = Image.open(img_path).convert('RGB')
                image = preprocess(image).unsqueeze(0).to(device)
            except Exception as e:
                print(f"Error loading {img_id}: {e}")
                continue
            
            # Get prediction
            coarse_logits, medium_logits, fine_logits = model(image)
            
            # Get predicted cell index
            _, coarse_pred = coarse_logits.max(1)
            coarse_cell_id = idx_to_coarse[coarse_pred.item()]
            
            # Get predicted coordinates (from cell center)
            pred_lat = cell_info['coarse'][coarse_cell_id]['lat']
            pred_lon = cell_info['coarse'][coarse_cell_id]['lon']
            
            # True coordinates
            true_lat = mapping[img_id]['lat']
            true_lon = mapping[img_id]['lon']
            
            # Calculate distance error
            dist_km = haversine_distance(true_lat, true_lon, pred_lat, pred_lon)
            distances.append(dist_km)
            
            results.append({
                'img_id': img_id,
                'true_lat': true_lat,
                'true_lon': true_lon,
                'pred_lat': pred_lat,
                'pred_lon': pred_lon,
                'distance_km': dist_km
            })
    
    # Calculate metrics
    distances = np.array(distances)
    
    print("\\n" + "="*70)
    print("RESULTS")
    print("="*70)
    
    print(f"\\n📊 Distance Metrics:")
    print(f"  Mean Error:     {distances.mean():.2f} km")
    print(f"  Median Error:   {np.median(distances):.2f} km")
    print(f"  Std Dev:        {distances.std():.2f} km")
    print(f"  Min Error:      {distances.min():.2f} km")
    print(f"  Max Error:      {distances.max():.2f} km")
    
    print(f"\\n🎯 Accuracy at Distance Thresholds:")
    thresholds = [1, 25, 200, 750, 2500]
    for threshold in thresholds:
        accuracy = (distances <= threshold).mean() * 100
        print(f"  Within {threshold:4d} km: {accuracy:5.2f}%")
    
    print(f"\\n📈 Distance Percentiles:")
    for p in [25, 50, 75, 90, 95, 99]:
        print(f"  {p}th percentile: {np.percentile(distances, p):7.2f} km")
    
    # Save results
    results_df = pd.DataFrame(results)
    results_df.to_csv('evaluation_results.csv', index=False)
    print(f"\\n💾 Results saved to: evaluation_results.csv")
    
    # Print sample predictions
    print(f"\\n📋 Sample Predictions (Best):")
    best_results = results_df.nsmallest(5, 'distance_km')
    print(best_results[['img_id', 'distance_km', 'true_lat', 'true_lon', 'pred_lat', 'pred_lon']].to_string(index=False))
    
    print(f"\\n📋 Sample Predictions (Worst):")
    worst_results = results_df.nlargest(5, 'distance_km')
    print(worst_results[['img_id', 'distance_km', 'true_lat', 'true_lon', 'pred_lat', 'pred_lon']].to_string(index=False))
    
    print("\\n" + "="*70)
    print("EVALUATION COMPLETE!")
    print("="*70)

if __name__ == "__main__":
    evaluate_model()
'''

with open('evaluate_pigeon.py', 'w') as f:
    f.write(eval_code_fixed)

print("✅ Fixed evaluation script created!")
print("\n🎯 Run evaluation:")
print("  !python evaluate_pigeon.py")

📊 Creating Fixed Evaluation Script...
✅ Fixed evaluation script created!

🎯 Run evaluation:
  !python evaluate_pigeon.py


In [13]:
!python evaluate_pigeon.py

PIGEON MODEL EVALUATION

Device: cuda

Loading checkpoint: pigeon_best_model.pth
Loading CLIP model...
✓ Model loaded (Epoch 15)

Loading test data...
✓ Data loaded: 237 images

RUNNING EVALUATION...
Evaluating: 100%|█████████████████████████████| 237/237 [00:05<00:00, 41.06it/s]

RESULTS

📊 Distance Metrics:
  Mean Error:     2568.49 km
  Median Error:   260.22 km
  Std Dev:        4438.24 km
  Min Error:      39.43 km
  Max Error:      17104.97 km

🎯 Accuracy at Distance Thresholds:
  Within    1 km:  0.00%
  Within   25 km:  0.00%
  Within  200 km: 34.60%
  Within  750 km: 66.67%
  Within 2500 km: 75.95%

📈 Distance Percentiles:
  25th percentile:  163.50 km
  50th percentile:  260.22 km
  75th percentile: 1895.82 km
  90th percentile: 10299.49 km
  95th percentile: 13761.85 km
  99th percentile: 16387.79 km

💾 Results saved to: evaluation_results.csv

📋 Sample Predictions (Best):
                                                  img_id  distance_km  true_lat    true_lon  pred_lat  

In [14]:
# Verify your data is still there
!ls data/
!ls *.pth

benchmarks	  im2gps3k_metadata.csv  image_to_geocell.json	test_images
cell_info.pkl	  im2gps_test.csv	 mp_16			train_images
geocells.pkl	  im2gps.zip		 README.md
google_landmarks  images_mp_16		 streetview_cropped
pigeon_best_model.pth	       pigeon_checkpoint_epoch15.pth
pigeon_checkpoint_epoch10.pth  pigeon_checkpoint_epoch5.pth


In [15]:
# ============================================================================
# IMPROVEMENT #1: DATA AUGMENTATION ONLY
# Safe, proven approach - no risky changes
# ============================================================================

print("📊 Creating Safe Data Augmentation Training Script...")

safe_augmentation_training = '''
"""
PIGEON Training with Data Augmentation ONLY
Conservative approach - only one change at a time
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import clip
import json
import pickle
import pandas as pd
from tqdm import tqdm
import os
import numpy as np
from s2sphere import CellId, LatLng
import torchvision.transforms as transforms

# ============================================================================
# Dataset with Augmentation
# ============================================================================

class AugmentedGeolocDataset(Dataset):
    """Dataset with conservative augmentation"""
    
    def __init__(self, image_dir, mapping_file, geocells_file, base_transform=None, augment=True):
        self.image_dir = image_dir
        self.base_transform = base_transform
        self.augment = augment
        
        with open(mapping_file, 'r') as f:
            self.mapping = json.load(f)
        
        with open(geocells_file, 'rb') as f:
            self.geocells = pickle.load(f)
        
        self.image_ids = list(self.mapping.keys())
        
        self.coarse_to_idx = {cell: idx for idx, cell in enumerate(sorted(self.geocells['coarse'].keys()))}
        self.medium_to_idx = {cell: idx for idx, cell in enumerate(sorted(self.geocells['medium'].keys()))}
        self.fine_to_idx = {cell: idx for idx, cell in enumerate(sorted(self.geocells['fine'].keys()))}
        
        # CONSERVATIVE augmentation - small changes only
        if augment:
            self.augmentation = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
                transforms.Resize(224),
            ])
        else:
            self.augmentation = transforms.Resize(224)
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.image_dir, img_id)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        # Apply conservative augmentation
        if self.augment:
            image = self.augmentation(image)
        
        # Then apply CLIP preprocessing
        if self.base_transform:
            image = self.base_transform(image)
        
        info = self.mapping[img_id]
        
        coarse_cell = info['coarse_cell']
        medium_cell = info['medium_cell']
        fine_cell = info['fine_cell']
        
        coarse_label = self.coarse_to_idx.get(coarse_cell, -1)
        medium_label = self.medium_to_idx.get(medium_cell, -1)
        fine_label = self.fine_to_idx.get(fine_cell, -1)
        
        return image, {
            'coarse': torch.tensor(coarse_label, dtype=torch.long),
            'medium': torch.tensor(medium_label, dtype=torch.long),
            'fine': torch.tensor(fine_label, dtype=torch.long),
            'lat': torch.tensor(info['lat'], dtype=torch.float),
            'lon': torch.tensor(info['lon'], dtype=torch.float)
        }

# ============================================================================
# Same Model Architecture as Baseline (NO changes)
# ============================================================================

class PIGEONModel(nn.Module):
    """SAME architecture as baseline - only training data changes"""
    
    def __init__(self, clip_model, num_coarse, num_medium, num_fine, embed_dim=512):
        super().__init__()
        self.clip_model = clip_model
        self.embed_dim = embed_dim
        
        # Freeze CLIP (same as baseline)
        for param in self.clip_model.parameters():
            param.requires_grad = False
        
        # SAME heads as baseline
        self.coarse_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_coarse)
        )
        
        self.medium_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_medium)
        )
        
        self.fine_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_fine)
        )
    
    def forward(self, images):
        with torch.no_grad():
            embeddings = self.clip_model.encode_image(images)
        
        embeddings = embeddings.float()
        
        coarse_logits = self.coarse_head(embeddings)
        medium_logits = self.medium_head(embeddings)
        fine_logits = self.fine_head(embeddings)
        
        return coarse_logits, medium_logits, fine_logits

# ============================================================================
# Training Function - ONLY Augmentation Added
# ============================================================================

def train_with_augmentation(
    num_epochs=15,
    batch_size=16,
    learning_rate=1e-3,
    train_split=0.8
):
    """Train with data augmentation - ONLY ONE CHANGE from baseline"""
    
    print("="*70)
    print("IMPROVEMENT #1: DATA AUGMENTATION")
    print("="*70)
    
    print("\\nChanges from baseline:")
    print("  ✓ Added data augmentation (flip + color jitter)")
    print("  ✗ NO architecture changes")
    print("  ✗ NO CLIP fine-tuning")
    print("  ✗ NO other modifications")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\\nDevice: {device}")
    
    # Load CLIP
    print("\\nLoading CLIP model...")
    clip_model, preprocess = clip.load("ViT-B/32", device=device)
    
    # Load geocells
    with open('data/geocells.pkl', 'rb') as f:
        geocells = pickle.load(f)
    
    num_coarse = len(geocells['coarse'])
    num_medium = len(geocells['medium'])
    num_fine = len(geocells['fine'])
    
    # Create model (SAME as baseline)
    print("\\nCreating model (same architecture as baseline)...")
    model = PIGEONModel(clip_model, num_coarse, num_medium, num_fine).to(device)
    
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Create dataset WITH augmentation
    print("\\nCreating augmented dataset...")
    full_dataset = AugmentedGeolocDataset(
        'data/test_images',
        'data/image_to_geocell.json',
        'data/geocells.pkl',
        base_transform=preprocess,
        augment=True  # THIS IS THE ONLY CHANGE!
    )
    
    # Split dataset (same split as baseline for fair comparison)
    dataset_size = len(full_dataset)
    train_size = int(train_split * dataset_size)
    val_size = dataset_size - train_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"Dataset split - Train: {train_size}, Val: {val_size}")
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # SAME optimizer and settings as baseline
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    
    print(f"\\nTraining configuration (same as baseline):")
    print(f"  Epochs: {num_epochs}")
    print(f"  Batch size: {batch_size}")
    print(f"  Learning rate: {learning_rate}")
    
    # Training loop (SAME as baseline)
    print("\\n" + "="*70)
    print("STARTING TRAINING")
    print("="*70)
    
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0
        train_coarse_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for images, labels in pbar:
            images = images.to(device)
            
            optimizer.zero_grad()
            
            coarse_logits, medium_logits, fine_logits = model(images)
            
            loss = 0
            valid_coarse = labels['coarse'] >= 0
            valid_medium = labels['medium'] >= 0
            valid_fine = labels['fine'] >= 0
            
            if valid_coarse.any():
                loss_coarse = criterion(coarse_logits[valid_coarse], labels['coarse'][valid_coarse].to(device))
                loss += loss_coarse
                
                _, pred = coarse_logits[valid_coarse].max(1)
                train_coarse_correct += pred.eq(labels['coarse'][valid_coarse].to(device)).sum().item()
                train_total += valid_coarse.sum().item()
            
            if valid_medium.any():
                loss += criterion(medium_logits[valid_medium], labels['medium'][valid_medium].to(device))
            
            if valid_fine.any():
                loss += criterion(fine_logits[valid_fine], labels['fine'][valid_fine].to(device))
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_coarse_correct / train_total if train_total > 0 else 0
        
        # Validation
        model.eval()
        val_loss = 0
        val_coarse_correct = 0
        val_total = 0
        
        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]  ")
            for images, labels in pbar:
                images = images.to(device)
                
                coarse_logits, medium_logits, fine_logits = model(images)
                
                loss = 0
                valid_coarse = labels['coarse'] >= 0
                valid_medium = labels['medium'] >= 0
                valid_fine = labels['fine'] >= 0
                
                if valid_coarse.any():
                    loss_coarse = criterion(coarse_logits[valid_coarse], labels['coarse'][valid_coarse].to(device))
                    loss += loss_coarse
                    
                    _, pred = coarse_logits[valid_coarse].max(1)
                    val_coarse_correct += pred.eq(labels['coarse'][valid_coarse].to(device)).sum().item()
                    val_total += valid_coarse.sum().item()
                
                if valid_medium.any():
                    loss += criterion(medium_logits[valid_medium], labels['medium'][valid_medium].to(device))
                
                if valid_fine.any():
                    loss += criterion(fine_logits[valid_fine], labels['fine'][valid_fine].to(device))
                
                val_loss += loss.item()
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_coarse_correct / val_total if val_total > 0 else 0
        
        scheduler.step()
        
        # Save history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"\\nEpoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
                'train_acc': train_acc,
                'val_acc': val_acc,
                'num_coarse': num_coarse,
                'num_medium': num_medium,
                'num_fine': num_fine,
                'history': history
            }, 'pigeon_augmented_best.pth')
            print(f"  ✓ Best model saved!")
        
        if (epoch + 1) % 5 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'history': history
            }, f'pigeon_augmented_epoch{epoch+1}.pth')
            print(f"  ✓ Checkpoint saved")
        
        print("-"*70)
    
    print("\\n" + "="*70)
    print("TRAINING COMPLETE!")
    print("="*70)
    print(f"Best validation loss: {best_val_loss:.4f}")
    print(f"Final validation accuracy: {history['val_acc'][-1]:.2f}%")
    print(f"\\nBaseline was: 35.29% val accuracy")
    print(f"Improvement: +{history['val_acc'][-1] - 35.29:.2f}%")
    
    # Save history
    import json
    with open('augmentation_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    
    return model, history

if __name__ == "__main__":
    model, history = train_with_augmentation()
'''

with open('train_augmentation_only.py', 'w') as f:
    f.write(safe_augmentation_training)

print("✅ Safe augmentation training script created!")
print("\n📄 File: train_augmentation_only.py")
print("\n🎯 This script:")
print("  ✓ Uses SAME architecture as baseline")
print("  ✓ Uses SAME hyperparameters")
print("  ✓ ONLY adds conservative data augmentation")
print("  ✓ Will definitely work (no NaN risk)")
print("\n🚀 To train: !python train_augmentation_only.py")

📊 Creating Safe Data Augmentation Training Script...
✅ Safe augmentation training script created!

📄 File: train_augmentation_only.py

🎯 This script:
  ✓ Uses SAME architecture as baseline
  ✓ Uses SAME hyperparameters
  ✓ ONLY adds conservative data augmentation
  ✓ Will definitely work (no NaN risk)

🚀 To train: !python train_augmentation_only.py


In [16]:
!python train_augmentation_only.py

IMPROVEMENT #1: DATA AUGMENTATION

Changes from baseline:
  ✓ Added data augmentation (flip + color jitter)
  ✗ NO architecture changes
  ✗ NO CLIP fine-tuning
  ✗ NO other modifications

Device: cuda

Loading CLIP model...

Creating model (same architecture as baseline)...
Trainable parameters: 419,427

Creating augmented dataset...
Dataset split - Train: 189, Val: 48

Training configuration (same as baseline):
  Epochs: 15
  Batch size: 16
  Learning rate: 0.001

STARTING TRAINING
Epoch 1/15 [Val]  : 100%|████████████| 3/3 [00:01<00:00,  2.41it/s, loss=9.6880]

Epoch 1/15:
  Train Loss: 10.3567 | Train Acc: 3.79%
  Val Loss:   10.2049 | Val Acc:   5.88%
  LR: 0.001000
  ✓ Best model saved!
----------------------------------------------------------------------
Epoch 2/15 [Val]  : 100%|████████████| 3/3 [00:01<00:00,  2.44it/s, loss=9.0451]

Epoch 2/15:
  Train Loss: 9.0980 | Train Acc: 28.03%
  Val Loss:   10.0633 | Val Acc:   5.88%
  LR: 0.001000
  ✓ Best model saved!
---------------

In [17]:
# ============================================================================
# IMPROVEMENT: Download Im2GPS3k Training Data
# Implements: "Data Updates" from mid-term proposal
# ============================================================================

print("📥 Creating Im2GPS3k Download Script...")
print("This implements your mid-term 'Data Updates' proposal")
print("="*70)

download_script = '''
"""
Download Im2GPS3k Training Images
Addresses mid-term proposal: Data Updates with contemporary images
"""

import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import os
from tqdm import tqdm
import time

def download_image(url, save_path, timeout=10):
    """Download single image from URL"""
    try:
        response = requests.get(url, timeout=timeout)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content))
            img = img.convert('RGB')
            img.save(save_path)
            return True
    except Exception as e:
        return False
    return False

def download_im2gps3k_images(metadata_file, output_dir, max_images=2000):
    """
    Download Im2GPS3k images from Flickr URLs
    
    Args:
        metadata_file: CSV with IMG_ID, LAT, LON
        output_dir: Where to save images
        max_images: Maximum images to download
    """
    
    print("="*70)
    print("DOWNLOADING IM2GPS3K TRAINING DATA")
    print("Mid-term Proposal: Data Updates")
    print("="*70)
    
    # Load metadata
    print(f"\\nLoading metadata from {metadata_file}...")
    df = pd.read_csv(metadata_file)
    print(f"Found {len(df)} images in metadata")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Limit to max_images
    if len(df) > max_images:
        print(f"Limiting to {max_images} images for speed")
        df = df.sample(n=max_images, random_state=42)
    
    # Download images
    successful = 0
    failed = 0
    
    print(f"\\nDownloading images to {output_dir}...")
    print("Note: Many URLs may be dead (Flickr images removed over time)")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Downloading"):
        img_id = row['IMG_ID']
        
        # Check if already downloaded
        save_path = os.path.join(output_dir, img_id)
        if os.path.exists(save_path):
            successful += 1
            continue
        
        # Try to construct Flickr URL from IMG_ID
        # Format: photoID_secret_size_userID@N00.jpg
        parts = img_id.replace('.jpg', '').split('_')
        if len(parts) >= 2:
            photo_id = parts[0]
            secret = parts[1]
            
            # Try different Flickr URL formats
            urls_to_try = [
                f"https://live.staticflickr.com/65535/{photo_id}_{secret}_b.jpg",
                f"https://farm1.staticflickr.com/{photo_id}_{secret}_b.jpg",
                f"https://farm2.staticflickr.com/{photo_id}_{secret}_b.jpg",
                f"https://live.staticflickr.com/{photo_id}_{secret}_b.jpg",
            ]
            
            success = False
            for url in urls_to_try:
                if download_image(url, save_path):
                    successful += 1
                    success = True
                    break
                time.sleep(0.1)  # Rate limiting
            
            if not success:
                failed += 1
        else:
            failed += 1
        
        # Progress update every 100 images
        if (successful + failed) % 100 == 0:
            success_rate = 100 * successful / (successful + failed)
            print(f"\\n  Downloaded: {successful}, Failed: {failed}, Success rate: {success_rate:.1f}%")
    
    # Final statistics
    print("\\n" + "="*70)
    print("DOWNLOAD COMPLETE")
    print("="*70)
    print(f"Successfully downloaded: {successful} images")
    print(f"Failed: {failed} images")
    print(f"Success rate: {100 * successful / (successful + failed):.1f}%")
    print(f"Images saved to: {output_dir}")
    
    # Filter metadata to only include successfully downloaded images
    downloaded_ids = [f for f in os.listdir(output_dir) if f.endswith('.jpg')]
    df_filtered = df[df['IMG_ID'].isin(downloaded_ids)]
    
    filtered_csv = metadata_file.replace('.csv', '_downloaded.csv')
    df_filtered.to_csv(filtered_csv, index=False)
    print(f"\\nFiltered metadata saved to: {filtered_csv}")
    print(f"Ready to use {len(df_filtered)} images for training!")
    
    return successful, failed

if __name__ == "__main__":
    # Download Im2GPS3k training images
    successful, failed = download_im2gps3k_images(
        metadata_file='data/im2gps3k_metadata.csv',
        output_dir='data/train_images_im2gps3k',
        max_images=2000  # Limit for speed (can increase if needed)
    )
'''

with open('download_im2gps3k.py', 'w') as f:
    f.write(download_script)

print("✅ Download script created!")
print("\n📄 File: download_im2gps3k.py")
print("\n🎯 This implements your mid-term proposal:")
print("  'Data Updates: Augment with contemporary images from Flickr'")
print("\n⏱️ Expected time: 1-2 hours")
print("📊 Expected result: 1000-1500 training images")
print("\n🚀 To start downloading:")
print("  !python download_im2gps3k.py")

📥 Creating Im2GPS3k Download Script...
This implements your mid-term 'Data Updates' proposal
✅ Download script created!

📄 File: download_im2gps3k.py

🎯 This implements your mid-term proposal:
  'Data Updates: Augment with contemporary images from Flickr'

⏱️ Expected time: 1-2 hours
📊 Expected result: 1000-1500 training images

🚀 To start downloading:
  !python download_im2gps3k.py


In [18]:
# Create the download script (run the code above first)
# Then:
!python download_im2gps3k.py

DOWNLOADING IM2GPS3K TRAINING DATA
Mid-term Proposal: Data Updates

Loading metadata from data/im2gps3k_metadata.csv...
Found 2997 images in metadata
Limiting to 2000 images for speed

Note: Many URLs may be dead (Flickr images removed over time)
Downloading:   5%|█▍                          | 99/2000 [00:44<13:16,  2.39it/s]
  Downloaded: 70, Failed: 30, Success rate: 70.0%
Downloading:  10%|██▋                        | 199/2000 [01:27<05:25,  5.54it/s]
  Downloaded: 137, Failed: 63, Success rate: 68.5%
Downloading:  15%|████                       | 299/2000 [02:20<27:12,  1.04it/s]
  Downloaded: 197, Failed: 103, Success rate: 65.7%
Downloading:  20%|█████▍                     | 399/2000 [03:09<20:15,  1.32it/s]
  Downloaded: 263, Failed: 137, Success rate: 65.8%
Downloading:  25%|██████▋                    | 499/2000 [04:01<03:57,  6.31it/s]
  Downloaded: 327, Failed: 173, Success rate: 65.4%
Downloading:  30%|████████                   | 599/2000 [04:42<08:05,  2.88it/s]
  Download

In [19]:
"""
PIGEON Dataset Verification Script
Checks integrity of Im2GPS + Im2GPS3k combined dataset before training
"""

import os
import pandas as pd
from PIL import Image
from pathlib import Path
import json

def verify_dataset():
    print("="*60)
    print("PIGEON DATASET VERIFICATION")
    print("="*60)
    
    # Paths
    base_path = Path("/kaggle/working/PIGEON")
    
    # Dataset 1: Original Im2GPS (189 images)
    original_test_csv = base_path / "data" / "im2gps_test.csv"
    
    # Dataset 2: New Im2GPS3k (1339 images)
    new_train_csv = base_path / "data" / "im2gps3k_metadata_downloaded.csv"
    new_train_images = base_path / "data" / "train_images_im2gps3k"
    
    # Geocell mappings
    geocells_file = base_path / "data" / "geocells.pkl"
    image_to_geocell_file = base_path / "data" / "image_to_geocell.json"
    
    print("\n1. CHECKING FILE EXISTENCE")
    print("-" * 60)
    
    files_to_check = {
        "Original test CSV": original_test_csv,
        "New training CSV": new_train_csv,
        "New training images folder": new_train_images,
        "Geocells structure": geocells_file,
        "Image→Geocell mappings": image_to_geocell_file
    }
    
    all_exist = True
    for name, path in files_to_check.items():
        exists = path.exists()
        status = "✓" if exists else "✗"
        print(f"{status} {name}: {path}")
        if not exists:
            all_exist = False
    
    if not all_exist:
        print("\n❌ ERROR: Missing required files!")
        return False
    
    print("\n2. LOADING METADATA")
    print("-" * 60)
    
    # Load CSVs
    df_original = pd.read_csv(original_test_csv)
    df_new = pd.read_csv(new_train_csv)
    
    print(f"Original dataset: {len(df_original)} images")
    print(f"New dataset: {len(df_new)} images")
    print(f"Combined total: {len(df_original) + len(df_new)} images")
    
    print("\n3. VERIFYING IMAGE FILES")
    print("-" * 60)
    
    # Check new training images
    actual_images = list(new_train_images.glob("*.jpg"))
    print(f"Images in folder: {len(actual_images)}")
    print(f"Images in CSV: {len(df_new)}")
    
    if len(actual_images) != len(df_new):
        print(f"⚠️  WARNING: Mismatch between folder ({len(actual_images)}) and CSV ({len(df_new)})")
    
    # Verify first 10 images are readable
    print("\nTesting image readability (first 10):")
    corrupted = []
    for i, row in df_new.head(10).iterrows():
        img_path = new_train_images / row['IMG_ID']
        try:
            img = Image.open(img_path)
            img.verify()  # Check if image is corrupted
            print(f"  ✓ {row['IMG_ID']}: {img.size}")
        except Exception as e:
            print(f"  ✗ {row['IMG_ID']}: CORRUPTED - {e}")
            corrupted.append(row['IMG_ID'])
    
    if corrupted:
        print(f"\n❌ ERROR: Found {len(corrupted)} corrupted images!")
        return False
    
    print("\n4. CHECKING GEOGRAPHIC DISTRIBUTION")
    print("-" * 60)
    
    # Check if both datasets have latitude/longitude
    if 'LAT' in df_original.columns and 'LON' in df_original.columns:
        print("Original dataset:")
        print(f"  Lat range: [{df_original['LAT'].min():.2f}, {df_original['LAT'].max():.2f}]")
        print(f"  Lon range: [{df_original['LON'].min():.2f}, {df_original['LON'].max():.2f}]")
    
    if 'LAT' in df_new.columns and 'LON' in df_new.columns:
        print("New dataset:")
        print(f"  Lat range: [{df_new['LAT'].min():.2f}, {df_new['LAT'].max():.2f}]")
        print(f"  Lon range: [{df_new['LON'].min():.2f}, {df_new['LON'].max():.2f}]")
    
    print("\n5. CHECKING GEOCELL MAPPINGS")
    print("-" * 60)
    
    with open(image_to_geocell_file, 'r') as f:
        image_to_geocell = json.load(f)
    
    print(f"Total images in geocell mapping: {len(image_to_geocell)}")
    
    # Check how many new images are mapped
    new_images_mapped = sum(1 for img_id in df_new['IMG_ID'] if img_id in image_to_geocell)
    print(f"New images with geocell mapping: {new_images_mapped}/{len(df_new)}")
    
    if new_images_mapped == 0:
        print("\n⚠️  WARNING: No new images have geocell mappings!")
        print("We need to regenerate geocells with the expanded dataset.")
        return "NEEDS_GEOCELL_REGENERATION"
    
    print("\n" + "="*60)
    print("VERIFICATION SUMMARY")
    print("="*60)
    print(f"✓ Total training images available: {len(df_original) + len(df_new)}")
    print(f"✓ All files exist and are readable")
    print(f"✓ Geographic coverage verified")
    
    if new_images_mapped < len(df_new):
        print(f"⚠️  Only {new_images_mapped}/{len(df_new)} new images have geocell mappings")
        print("   Next step: Regenerate geocells with expanded dataset")
        return "NEEDS_GEOCELL_REGENERATION"
    else:
        print(f"✓ All images have geocell mappings")
        print("\n🎯 READY TO TRAIN!")
        return True

if __name__ == "__main__":
    result = verify_dataset()
    
    if result == "NEEDS_GEOCELL_REGENERATION":
        print("\n📋 NEXT STEP: Run geocell regeneration script")
    elif result == True:
        print("\n📋 NEXT STEP: Run training script")
    else:
        print("\n❌ Please fix errors before proceeding")

PIGEON DATASET VERIFICATION

1. CHECKING FILE EXISTENCE
------------------------------------------------------------
✓ Original test CSV: /kaggle/working/PIGEON/data/im2gps_test.csv
✓ New training CSV: /kaggle/working/PIGEON/data/im2gps3k_metadata_downloaded.csv
✓ New training images folder: /kaggle/working/PIGEON/data/train_images_im2gps3k
✓ Geocells structure: /kaggle/working/PIGEON/data/geocells.pkl
✓ Image→Geocell mappings: /kaggle/working/PIGEON/data/image_to_geocell.json

2. LOADING METADATA
------------------------------------------------------------
Original dataset: 237 images
New dataset: 1339 images
Combined total: 1576 images

3. VERIFYING IMAGE FILES
------------------------------------------------------------
Images in folder: 1339
Images in CSV: 1339

Testing image readability (first 10):
  ✓ 292993389_9b2f509aeb_111_63163416@N00.jpg: (800, 600)
  ✓ 235419360_93d534a2f5_94_82927779@N00.jpg: (1024, 681)
  ✓ 1125852220_b13d685cea_1111_93455345@N00.jpg: (768, 1024)
  ✓ 1064

In [20]:
"""
PIGEON Geocell Regeneration Script
Rebuilds hierarchical geocells with expanded Im2GPS + Im2GPS3k dataset
"""

import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
from collections import defaultdict
import s2sphere
from tqdm import tqdm

def lat_lon_to_s2_cell(lat, lon, level):
    """Convert latitude/longitude to S2 cell at specified level"""
    latlng = s2sphere.LatLng.from_degrees(lat, lon)
    cell_id = s2sphere.CellId.from_lat_lng(latlng).parent(level)
    return cell_id.to_token()

def create_hierarchical_geocells(df, min_images_per_cell=3):
    """
    Create 3-level hierarchical geocell structure
    
    Args:
        df: DataFrame with LAT, LON, IMG_ID columns
        min_images_per_cell: Minimum images required per cell
    
    Returns:
        geocells: Dict mapping level → list of cell tokens
        image_to_geocell: Dict mapping IMG_ID → (coarse, medium, fine) cells
        cell_info: Dict with cell coordinates and image counts
    """
    
    print("Creating hierarchical geocells...")
    print(f"Total images: {len(df)}")
    print(f"Minimum images per cell: {min_images_per_cell}")
    
    # S2 levels for hierarchy (same as baseline)
    # Level 4: ~600km cells (coarse)
    # Level 6: ~150km cells (medium)  
    # Level 8: ~40km cells (fine)
    coarse_level = 4
    medium_level = 6
    fine_level = 8
    
    # Step 1: Assign each image to S2 cells at all levels
    print("\n1. Assigning images to S2 cells...")
    df['coarse_cell'] = df.apply(lambda row: lat_lon_to_s2_cell(row['LAT'], row['LON'], coarse_level), axis=1)
    df['medium_cell'] = df.apply(lambda row: lat_lon_to_s2_cell(row['LAT'], row['LON'], medium_level), axis=1)
    df['fine_cell'] = df.apply(lambda row: lat_lon_to_s2_cell(row['LAT'], row['LON'], fine_level), axis=1)
    
    # Step 2: Count images per cell at each level
    coarse_counts = df['coarse_cell'].value_counts()
    medium_counts = df['medium_cell'].value_counts()
    fine_counts = df['fine_cell'].value_counts()
    
    print(f"   Coarse cells (L{coarse_level}): {len(coarse_counts)} unique cells")
    print(f"   Medium cells (L{medium_level}): {len(medium_counts)} unique cells")
    print(f"   Fine cells (L{fine_level}): {len(fine_counts)} unique cells")
    
    # Step 3: Filter cells with sufficient images
    print(f"\n2. Filtering cells (min {min_images_per_cell} images)...")
    valid_coarse = set(coarse_counts[coarse_counts >= min_images_per_cell].index)
    valid_medium = set(medium_counts[medium_counts >= min_images_per_cell].index)
    valid_fine = set(fine_counts[fine_counts >= min_images_per_cell].index)
    
    print(f"   Valid coarse cells: {len(valid_coarse)}")
    print(f"   Valid medium cells: {len(valid_medium)}")
    print(f"   Valid fine cells: {len(valid_fine)}")
    
    # Step 4: Create geocell structure
    geocells = {
        'coarse': sorted(list(valid_coarse)),
        'medium': sorted(list(valid_medium)),
        'fine': sorted(list(valid_fine))
    }
    
    # Step 5: Map images to valid geocells
    print("\n3. Mapping images to geocells...")
    image_to_geocell = {}
    valid_images = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        img_id = row['IMG_ID']
        coarse = row['coarse_cell'] if row['coarse_cell'] in valid_coarse else None
        medium = row['medium_cell'] if row['medium_cell'] in valid_medium else None
        fine = row['fine_cell'] if row['fine_cell'] in valid_fine else None
        
        # Image must have at least coarse cell mapping
        if coarse is not None:
            image_to_geocell[img_id] = {
                'coarse': coarse,
                'medium': medium,
                'fine': fine,
                'lat': float(row['LAT']),
                'lon': float(row['LON'])
            }
            valid_images += 1
    
    print(f"   Images with valid mappings: {valid_images}/{len(df)}")
    
    # Step 6: Calculate cell centers and info
    print("\n4. Calculating cell information...")
    cell_info = {
        'coarse': {},
        'medium': {},
        'fine': {}
    }
    
    for level_name, level_num in [('coarse', coarse_level), ('medium', medium_level), ('fine', fine_level)]:
        for cell_token in geocells[level_name]:
            cell_id = s2sphere.CellId.from_token(cell_token)
            center = cell_id.to_lat_lng()
            
            # Count images in this cell
            img_count = sum(1 for img_data in image_to_geocell.values() 
                          if img_data.get(level_name) == cell_token)
            
            cell_info[level_name][cell_token] = {
                'lat': center.lat().degrees,
                'lon': center.lng().degrees,
                'image_count': img_count,
                'level': level_num
            }
    
    return geocells, image_to_geocell, cell_info

def save_geocells(geocells, image_to_geocell, cell_info, output_dir):
    """Save geocell structures to disk"""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save geocells structure
    with open(output_dir / 'geocells.pkl', 'wb') as f:
        pickle.dump(geocells, f)
    print(f"✓ Saved: {output_dir / 'geocells.pkl'}")
    
    # Save image to geocell mappings
    with open(output_dir / 'image_to_geocell.json', 'w') as f:
        json.dump(image_to_geocell, f, indent=2)
    print(f"✓ Saved: {output_dir / 'image_to_geocell.json'}")
    
    # Save cell info
    with open(output_dir / 'cell_info.pkl', 'wb') as f:
        pickle.dump(cell_info, f)
    print(f"✓ Saved: {output_dir / 'cell_info.pkl'}")

def main():
    print("="*70)
    print("PIGEON GEOCELL REGENERATION - EXPANDED DATASET")
    print("="*70)
    
    base_path = Path("/kaggle/working/PIGEON")
    
    # Load both datasets
    print("\nLoading datasets...")
    df_im2gps = pd.read_csv(base_path / "data" / "im2gps_test.csv")
    df_im2gps3k = pd.read_csv(base_path / "data" / "im2gps3k_metadata_downloaded.csv")
    
    print(f"  Im2GPS: {len(df_im2gps)} images")
    print(f"  Im2GPS3k: {len(df_im2gps3k)} images")
    
    # Combine datasets
    df_combined = pd.concat([df_im2gps, df_im2gps3k], ignore_index=True)
    print(f"  Combined: {len(df_combined)} images")
    
    # Verify required columns
    required_cols = ['IMG_ID', 'LAT', 'LON']
    missing_cols = [col for col in required_cols if col not in df_combined.columns]
    if missing_cols:
        print(f"\n❌ ERROR: Missing columns: {missing_cols}")
        return
    
    # Remove any rows with missing coordinates
    df_clean = df_combined.dropna(subset=['LAT', 'LON'])
    if len(df_clean) < len(df_combined):
        print(f"  Removed {len(df_combined) - len(df_clean)} images with missing coordinates")
    
    print(f"  Final dataset: {len(df_clean)} images with valid coordinates")
    
    # Generate geocells
    print("\n" + "="*70)
    geocells, image_to_geocell, cell_info = create_hierarchical_geocells(
        df_clean, 
        min_images_per_cell=3
    )
    
    # Print summary statistics
    print("\n" + "="*70)
    print("GEOCELL STRUCTURE SUMMARY")
    print("="*70)
    print(f"Coarse cells (L4, ~600km): {len(geocells['coarse'])} cells")
    print(f"Medium cells (L6, ~150km): {len(geocells['medium'])} cells")
    print(f"Fine cells (L8, ~40km): {len(geocells['fine'])} cells")
    print(f"\nTotal images mapped: {len(image_to_geocell)}/{len(df_clean)}")
    
    # Calculate coverage statistics
    images_with_fine = sum(1 for img in image_to_geocell.values() if img['fine'] is not None)
    images_with_medium = sum(1 for img in image_to_geocell.values() if img['medium'] is not None)
    
    print(f"\nCoverage:")
    print(f"  Coarse only: {len(image_to_geocell) - images_with_medium} images")
    print(f"  Medium: {images_with_medium} images")
    print(f"  Fine: {images_with_fine} images")
    
    # Save results
    print("\n" + "="*70)
    print("SAVING RESULTS")
    print("="*70)
    save_geocells(geocells, image_to_geocell, cell_info, base_path / "data")
    
    print("\n✅ GEOCELL REGENERATION COMPLETE!")
    print("\n📋 NEXT STEP: Run training script with expanded dataset")
    print("="*70)

if __name__ == "__main__":
    main()

PIGEON GEOCELL REGENERATION - EXPANDED DATASET

Loading datasets...
  Im2GPS: 237 images
  Im2GPS3k: 1339 images
  Combined: 1576 images
  Final dataset: 1576 images with valid coordinates

Creating hierarchical geocells...
Total images: 1576
Minimum images per cell: 3

1. Assigning images to S2 cells...
   Coarse cells (L4): 189 unique cells
   Medium cells (L6): 420 unique cells
   Fine cells (L8): 605 unique cells

2. Filtering cells (min 3 images)...
   Valid coarse cells: 107
   Valid medium cells: 150
   Valid fine cells: 135

3. Mapping images to geocells...


100%|██████████| 1576/1576 [00:00<00:00, 16780.96it/s]

   Images with valid mappings: 1463/1576

4. Calculating cell information...

GEOCELL STRUCTURE SUMMARY
Coarse cells (L4, ~600km): 107 cells
Medium cells (L6, ~150km): 150 cells
Fine cells (L8, ~40km): 135 cells

Total images mapped: 1462/1576

Coverage:
  Coarse only: 236 images
  Medium: 1226 images
  Fine: 997 images

SAVING RESULTS
✓ Saved: /kaggle/working/PIGEON/data/geocells.pkl
✓ Saved: /kaggle/working/PIGEON/data/image_to_geocell.json
✓ Saved: /kaggle/working/PIGEON/data/cell_info.pkl

✅ GEOCELL REGENERATION COMPLETE!

📋 NEXT STEP: Run training script with expanded dataset


In [21]:
"""
PIGEON Training Script - Expanded Dataset
Trains on combined Im2GPS + Im2GPS3k (1462 images)
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
from tqdm import tqdm
import clip
import time

# Configuration
CONFIG = {
    'batch_size': 16,
    'num_epochs': 15,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 2,
    'clip_model': 'ViT-B/32'
}

class PigeonDataset(Dataset):
    """Dataset for PIGEON training with hierarchical geocells"""
    
    def __init__(self, image_paths, image_to_geocell, geocells, transform=None):
        """
        Args:
            image_paths: List of (img_id, full_path) tuples
            image_to_geocell: Dict mapping img_id to geocell info
            geocells: Dict with coarse/medium/fine cell lists
            transform: Image transformations
        """
        self.image_paths = image_paths
        self.image_to_geocell = image_to_geocell
        self.transform = transform
        
        # Create cell to index mappings
        self.coarse_to_idx = {cell: idx for idx, cell in enumerate(geocells['coarse'])}
        self.medium_to_idx = {cell: idx for idx, cell in enumerate(geocells['medium'])}
        self.fine_to_idx = {cell: idx for idx, cell in enumerate(geocells['fine'])}
        
        # Filter to only images with valid geocell mappings
        self.valid_samples = []
        for img_id, img_path in image_paths:
            if img_id in image_to_geocell:
                geocell_info = image_to_geocell[img_id]
                if geocell_info['coarse'] is not None:  # Must have at least coarse
                    self.valid_samples.append((img_id, img_path, geocell_info))
        
        print(f"  Dataset: {len(self.valid_samples)}/{len(image_paths)} images with valid mappings")
    
    def __len__(self):
        return len(self.valid_samples)
    
    def __getitem__(self, idx):
        img_id, img_path, geocell_info = self.valid_samples[idx]
        
        # Load and transform image
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        
        # Get labels (use -1 for missing medium/fine cells)
        coarse_label = self.coarse_to_idx[geocell_info['coarse']]
        medium_label = self.medium_to_idx.get(geocell_info['medium'], -1) if geocell_info['medium'] else -1
        fine_label = self.fine_to_idx.get(geocell_info['fine'], -1) if geocell_info['fine'] else -1
        
        return image, coarse_label, medium_label, fine_label

class PigeonModel(nn.Module):
    """PIGEON model with frozen CLIP backbone and hierarchical classification heads"""
    
    def __init__(self, num_coarse, num_medium, num_fine, clip_model_name='ViT-B/32'):
        super().__init__()
        
        # Load CLIP model and freeze it
        self.clip_model, _ = clip.load(clip_model_name, device='cpu')
        for param in self.clip_model.parameters():
            param.requires_grad = False
        
        # Get CLIP embedding dimension
        clip_embed_dim = 512  # ViT-B/32
        
        # Hierarchical classification heads
        self.coarse_head = nn.Sequential(
            nn.Linear(clip_embed_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_coarse)
        )
        
        self.medium_head = nn.Sequential(
            nn.Linear(clip_embed_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_medium)
        )
        
        self.fine_head = nn.Sequential(
            nn.Linear(clip_embed_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_fine)
        )
    
    def forward(self, images):
        # Extract CLIP features
        with torch.no_grad():
            features = self.clip_model.encode_image(images).float()
        
        # Apply classification heads
        coarse_logits = self.coarse_head(features)
        medium_logits = self.medium_head(features)
        fine_logits = self.fine_head(features)
        
        return coarse_logits, medium_logits, fine_logits

def hierarchical_loss(coarse_logits, medium_logits, fine_logits, 
                     coarse_labels, medium_labels, fine_labels):
    """
    Hierarchical cross-entropy loss
    Only computes loss for valid labels (non-negative)
    """
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    
    loss_coarse = criterion(coarse_logits, coarse_labels)
    loss_medium = criterion(medium_logits, medium_labels)
    loss_fine = criterion(fine_logits, fine_labels)
    
    # Weight losses (coarse is most important)
    total_loss = 0.5 * loss_coarse + 0.3 * loss_medium + 0.2 * loss_fine
    
    return total_loss, loss_coarse, loss_medium, loss_fine

def train_epoch(model, dataloader, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    coarse_correct = 0
    total_samples = 0
    
    pbar = tqdm(dataloader, desc='Training')
    for images, coarse_labels, medium_labels, fine_labels in pbar:
        images = images.to(device)
        coarse_labels = coarse_labels.to(device)
        medium_labels = medium_labels.to(device)
        fine_labels = fine_labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        coarse_logits, medium_logits, fine_logits = model(images)
        
        # Compute loss
        loss, loss_c, loss_m, loss_f = hierarchical_loss(
            coarse_logits, medium_logits, fine_logits,
            coarse_labels, medium_labels, fine_labels
        )
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        coarse_correct += (coarse_logits.argmax(1) == coarse_labels).sum().item()
        total_samples += len(images)
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100*coarse_correct/total_samples:.1f}%'
        })
    
    return total_loss / len(dataloader), coarse_correct / total_samples

def validate(model, dataloader, device):
    """Validate model"""
    model.eval()
    total_loss = 0
    coarse_correct = 0
    medium_correct = 0
    fine_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for images, coarse_labels, medium_labels, fine_labels in tqdm(dataloader, desc='Validation'):
            images = images.to(device)
            coarse_labels = coarse_labels.to(device)
            medium_labels = medium_labels.to(device)
            fine_labels = fine_labels.to(device)
            
            # Forward pass
            coarse_logits, medium_logits, fine_logits = model(images)
            
            # Compute loss
            loss, _, _, _ = hierarchical_loss(
                coarse_logits, medium_logits, fine_logits,
                coarse_labels, medium_labels, fine_labels
            )
            
            total_loss += loss.item()
            
            # Track accuracy at each level
            coarse_correct += (coarse_logits.argmax(1) == coarse_labels).sum().item()
            
            valid_medium = medium_labels >= 0
            if valid_medium.any():
                medium_correct += ((medium_logits.argmax(1) == medium_labels) & valid_medium).sum().item()
            
            valid_fine = fine_labels >= 0
            if valid_fine.any():
                fine_correct += ((fine_logits.argmax(1) == fine_labels) & valid_fine).sum().item()
            
            total_samples += len(images)
    
    return {
        'loss': total_loss / len(dataloader),
        'coarse_acc': coarse_correct / total_samples,
        'medium_acc': medium_correct / total_samples if total_samples > 0 else 0,
        'fine_acc': fine_correct / total_samples if total_samples > 0 else 0
    }

def main():
    print("="*70)
    print("PIGEON TRAINING - EXPANDED DATASET")
    print("="*70)
    print(f"Device: {CONFIG['device']}")
    print(f"Batch size: {CONFIG['batch_size']}")
    print(f"Epochs: {CONFIG['num_epochs']}")
    print(f"Learning rate: {CONFIG['learning_rate']}")
    
    base_path = Path("/kaggle/working/PIGEON")
    
    # Load geocell structure
    print("\nLoading geocells...")
    with open(base_path / "data" / "geocells.pkl", 'rb') as f:
        geocells = pickle.load(f)
    
    with open(base_path / "data" / "image_to_geocell.json", 'r') as f:
        image_to_geocell = json.load(f)
    
    print(f"  Coarse cells: {len(geocells['coarse'])}")
    print(f"  Medium cells: {len(geocells['medium'])}")
    print(f"  Fine cells: {len(geocells['fine'])}")
    print(f"  Mapped images: {len(image_to_geocell)}")
    
    # Load image paths from both datasets
    print("\nLoading image paths...")
    
    # Im2GPS dataset
    df_im2gps = pd.read_csv(base_path / "data" / "im2gps_test.csv")
    im2gps_paths = [(row['IMG_ID'], base_path / "data" / "test_images" / row['IMG_ID']) 
                    for _, row in df_im2gps.iterrows()]
    
    # Im2GPS3k dataset
    df_im2gps3k = pd.read_csv(base_path / "data" / "im2gps3k_metadata_downloaded.csv")
    im2gps3k_paths = [(row['IMG_ID'], base_path / "data" / "train_images_im2gps3k" / row['IMG_ID']) 
                      for _, row in df_im2gps3k.iterrows()]
    
    # Combine and filter to images with geocell mappings
    all_paths = im2gps_paths + im2gps3k_paths
    valid_paths = [(img_id, path) for img_id, path in all_paths if img_id in image_to_geocell]
    
    print(f"  Total images: {len(all_paths)}")
    print(f"  With geocell mappings: {len(valid_paths)}")
    
    # Train/validation split (80/20)
    np.random.seed(42)
    indices = np.random.permutation(len(valid_paths))
    split_idx = int(0.8 * len(valid_paths))
    
    train_paths = [valid_paths[i] for i in indices[:split_idx]]
    val_paths = [valid_paths[i] for i in indices[split_idx:]]
    
    print(f"  Training: {len(train_paths)} images")
    print(f"  Validation: {len(val_paths)} images")
    
    # CLIP preprocessing
    _, preprocess = clip.load(CONFIG['clip_model'], device='cpu')
    
    # Create datasets
    print("\nCreating datasets...")
    train_dataset = PigeonDataset(train_paths, image_to_geocell, geocells, transform=preprocess)
    val_dataset = PigeonDataset(val_paths, image_to_geocell, geocells, transform=preprocess)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    
    # Create model
    print("\nInitializing model...")
    model = PigeonModel(
        num_coarse=len(geocells['coarse']),
        num_medium=len(geocells['medium']),
        num_fine=len(geocells['fine'])
    ).to(CONFIG['device'])
    
    # Count trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Total parameters: {total_params:,}")
    
    # Optimizer
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )
    
    # Training loop
    print("\n" + "="*70)
    print("TRAINING START")
    print("="*70)
    
    best_val_acc = 0
    best_model_path = base_path / "pigeon_expanded_best.pth"
    
    for epoch in range(CONFIG['num_epochs']):
        print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
        print("-" * 70)
        
        start_time = time.time()
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, CONFIG['device'])
        
        # Validate
        val_metrics = validate(model, val_loader, CONFIG['device'])
        
        epoch_time = time.time() - start_time
        
        # Print results
        print(f"\nEpoch {epoch+1} Results:")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {100*train_acc:.2f}%")
        print(f"  Val Loss: {val_metrics['loss']:.4f}")
        print(f"  Val Coarse Acc: {100*val_metrics['coarse_acc']:.2f}%")
        print(f"  Val Medium Acc: {100*val_metrics['medium_acc']:.2f}%")
        print(f"  Val Fine Acc: {100*val_metrics['fine_acc']:.2f}%")
        print(f"  Time: {epoch_time:.1f}s")
        
        # Save best model
        if val_metrics['coarse_acc'] > best_val_acc:
            best_val_acc = val_metrics['coarse_acc']
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': best_val_acc,
                'geocells': geocells,
                'config': CONFIG
            }, best_model_path)
            print(f"  ✓ Saved best model (val_acc: {100*best_val_acc:.2f}%)")
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE!")
    print("="*70)
    print(f"Best validation accuracy: {100*best_val_acc:.2f}%")
    print(f"Model saved: {best_model_path}")
    print("\n📋 NEXT STEP: Evaluate model on test set")

if __name__ == "__main__":
    main()

PIGEON TRAINING - EXPANDED DATASET
Device: cuda
Batch size: 16
Epochs: 15
Learning rate: 0.001

Loading geocells...
  Coarse cells: 107
  Medium cells: 150
  Fine cells: 135
  Mapped images: 1462

Loading image paths...
  Total images: 1576
  With geocell mappings: 1463
  Training: 1170 images
  Validation: 293 images

Creating datasets...
  Dataset: 1170/1170 images with valid mappings
  Dataset: 293/293 images with valid mappings

Initializing model...
  Trainable parameters: 989,064
  Total parameters: 152,266,377

TRAINING START

Epoch 1/15
----------------------------------------------------------------------


Validation: 100%|██████████| 19/19 [00:02<00:00,  7.41it/s]



Epoch 1 Results:
  Train Loss: 4.1518 | Train Acc: 17.26%
  Val Loss: 3.8055
  Val Coarse Acc: 25.94%
  Val Medium Acc: 15.36%
  Val Fine Acc: 14.33%
  Time: 12.5s
  ✓ Saved best model (val_acc: 25.94%)

Epoch 2/15
----------------------------------------------------------------------


Validation: 100%|██████████| 19/19 [00:02<00:00,  7.45it/s]



Epoch 2 Results:
  Train Loss: 2.9263 | Train Acc: 36.15%
  Val Loss: 3.4119
  Val Coarse Acc: 34.81%
  Val Medium Acc: 24.23%
  Val Fine Acc: 21.16%
  Time: 12.1s
  ✓ Saved best model (val_acc: 34.81%)

Epoch 3/15
----------------------------------------------------------------------


...


Epoch 13 Results:
  Train Loss: 0.0888 | Train Acc: 99.32%
  Val Loss: 3.2769
  Val Coarse Acc: 40.61%
  Val Medium Acc: 26.96%
  Val Fine Acc: 27.99%
  Time: 11.8s

Epoch 14/15
----------------------------------------------------------------------


Validation: 100%|██████████| 19/19 [00:02<00:00,  7.55it/s]



Epoch 14 Results:
  Train Loss: 0.0796 | Train Acc: 99.57%
  Val Loss: 3.3037
  Val Coarse Acc: 39.93%
  Val Medium Acc: 30.03%
  Val Fine Acc: 26.28%
  Time: 12.0s

Epoch 15/15
----------------------------------------------------------------------


Validation: 100%|██████████| 19/19 [00:02<00:00,  7.72it/s]


Epoch 15 Results:
  Train Loss: 0.0832 | Train Acc: 98.97%
  Val Loss: 3.3085
  Val Coarse Acc: 39.59%
  Val Medium Acc: 28.33%
  Val Fine Acc: 26.62%
  Time: 12.0s

TRAINING COMPLETE!
Best validation accuracy: 40.61%
Model saved: /kaggle/working/PIGEON/pigeon_expanded_best.pth

📋 NEXT STEP: Evaluate model on test set


In [22]:
"""
PIGEON Data Prep Script
1. Verifies Im2GPS3k download integrity
2. Regenerates geocells using COMBINED dataset (Original + New)
"""

import pandas as pd
import numpy as np
import pickle
import json
import os
from pathlib import Path
from s2sphere import CellId, LatLng
from tqdm import tqdm
from PIL import Image

def regenerate_geocells():
    print("="*70)
    print("PIGEON DATA PREP: VERIFY & REGENERATE")
    print("="*70)

    base_path = Path("data")
    
    # --- 1. Load & Verify Data ---
    print("\n1. Loading Datasets...")
    
    # Original Data (Im2GPS)
    try:
        df_orig = pd.read_csv(base_path / "im2gps_test.csv")
        # Fix paths for original data
        df_orig['full_path'] = df_orig['IMG_ID'].apply(lambda x: base_path / "test_images" / x)
        print(f"   Original Im2GPS: {len(df_orig)} images")
    except Exception as e:
        print(f"   ❌ Error loading original data: {e}")
        return

    # New Data (Im2GPS3k)
    try:
        df_new = pd.read_csv(base_path / "im2gps3k_metadata_downloaded.csv")
        # Fix paths for new data
        df_new['full_path'] = df_new['IMG_ID'].apply(lambda x: base_path / "train_images_im2gps3k" / x)
        print(f"   New Im2GPS3k:    {len(df_new)} images")
    except Exception as e:
        print(f"   ❌ Error loading new data: {e}")
        return

    # Combine
    df_combined = pd.concat([df_orig, df_new], ignore_index=True)
    print(f"   TOTAL COMBINED:  {len(df_combined)} images")

    # Verify images exist
    print("\n2. Verifying image files...")
    valid_mask = []
    for path in tqdm(df_combined['full_path'], desc="Checking files"):
        valid_mask.append(path.exists())
    
    df_combined = df_combined[valid_mask].copy()
    print(f"   ✓ Valid images ready for training: {len(df_combined)}")

    # --- 2. Regenerate Geocells ---
    print("\n3. Regenerating Geocells (S2 Geometry)...")
    
    # Parameters (Same as baseline)
    coarse_level = 4  # ~1000km
    medium_level = 6  # ~200km
    fine_level = 8    # ~50km
    min_images = 3    # Min images to form a class
    
    geocells = {'coarse': [], 'medium': [], 'fine': []}
    image_to_geocell = {}
    cell_info = {'coarse': {}, 'medium': {}, 'fine': {}}
    
    # Calculate cell IDs for all images
    print("   Calculating S2 cell IDs...")
    for idx, row in df_combined.iterrows():
        lat, lon = row['LAT'], row['LON']
        latlng = LatLng.from_degrees(lat, lon)
        
        df_combined.at[idx, 'c_id'] = CellId.from_lat_lng(latlng).parent(coarse_level).to_token()
        df_combined.at[idx, 'm_id'] = CellId.from_lat_lng(latlng).parent(medium_level).to_token()
        df_combined.at[idx, 'f_id'] = CellId.from_lat_lng(latlng).parent(fine_level).to_token()

    # Filter cells by frequency
    print("   Filtering sparse cells...")
    for level, col in [('coarse', 'c_id'), ('medium', 'm_id'), ('fine', 'f_id')]:
        counts = df_combined[col].value_counts()
        valid_cells = counts[counts >= min_images].index.tolist()
        geocells[level] = sorted(valid_cells)
        
        # Store cell center info
        for cell_token in valid_cells:
            cell = CellId.from_token(cell_token)
            ll = cell.to_lat_lng()
            cell_info[level][cell_token] = {
                'lat': ll.lat().degrees,
                'lon': ll.lng().degrees,
                'count': int(counts[cell_token])
            }
            
    print(f"   Stats: {len(geocells['coarse'])} Coarse, {len(geocells['medium'])} Medium, {len(geocells['fine'])} Fine cells")

    # Map images to valid cells
    print("   Mapping images to valid cells...")
    for idx, row in df_combined.iterrows():
        img_id = row['IMG_ID']
        c_token = row['c_id']
        m_token = row['m_id']
        f_token = row['f_id']
        
        # Only keep if coarse cell is valid (minimum requirement)
        if c_token in geocells['coarse']:
            mapping = {
                'lat': row['LAT'],
                'lon': row['LON'],
                'coarse': c_token,
                'medium': m_token if m_token in geocells['medium'] else None,
                'fine': f_token if f_token in geocells['fine'] else None,
                'split': 'train' if 'train_images' in str(row['full_path']) else 'test'
            }
            image_to_geocell[img_id] = mapping

    # --- 3. Save ---
    print("\n4. Saving updated structures...")
    with open(base_path / 'geocells.pkl', 'wb') as f:
        pickle.dump(geocells, f)
    
    with open(base_path / 'image_to_geocell.json', 'w') as f:
        json.dump(image_to_geocell, f, indent=2)
        
    with open(base_path / 'cell_info.pkl', 'wb') as f:
        pickle.dump(cell_info, f)

    print("="*70)
    print("✅ PREP COMPLETE! Ready for 'train_expanded.py'")
    print("="*70)

if __name__ == "__main__":
    regenerate_geocells()

PIGEON DATA PREP: VERIFY & REGENERATE

1. Loading Datasets...
   Original Im2GPS: 237 images
   New Im2GPS3k:    1339 images
   TOTAL COMBINED:  1576 images

2. Verifying image files...


Checking files: 100%|██████████| 1576/1576 [00:00<00:00, 77678.69it/s]

   ✓ Valid images ready for training: 1576

3. Regenerating Geocells (S2 Geometry)...
   Calculating S2 cell IDs...


   Filtering sparse cells...
   Stats: 107 Coarse, 150 Medium, 135 Fine cells
   Mapping images to valid cells...

4. Saving updated structures...
✅ PREP COMPLETE! Ready for 'train_expanded.py'


In [23]:
"""
PIGEON Expanded Training Script
Trains on 1500+ images (Im2GPS + Im2GPS3k)
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import clip
import json
import pickle
import pandas as pd
from tqdm import tqdm
import os
import numpy as np

# --- CONFIG ---
BATCH_SIZE = 16
EPOCHS = 20  # Increased slightly for more data
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 STARTING EXPANDED TRAINING on {DEVICE}")

# --- 1. Dataset Class ---
class ExpandedGeolocDataset(Dataset):
    def __init__(self, transform=None):
        self.transform = transform
        self.base_path = "data"
        
        # Load mappings
        with open(os.path.join(self.base_path, 'image_to_geocell.json'), 'r') as f:
            self.mapping = json.load(f)
            
        with open(os.path.join(self.base_path, 'geocells.pkl'), 'rb') as f:
            self.geocells = pickle.load(f)
            
        self.image_ids = list(self.mapping.keys())
        
        # Create class indices
        self.c_to_idx = {c: i for i, c in enumerate(self.geocells['coarse'])}
        self.m_to_idx = {c: i for i, c in enumerate(self.geocells['medium'])}
        self.f_to_idx = {c: i for i, c in enumerate(self.geocells['fine'])}
        
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        data = self.mapping[img_id]
        
        # Determine path (Original vs New)
        if data['split'] == 'test':
            path = os.path.join(self.base_path, 'test_images', img_id)
        else:
            path = os.path.join(self.base_path, 'train_images_im2gps3k', img_id)
            
        try:
            image = Image.open(path).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except:
            # Fallback for corrupt images
            image = torch.zeros((3, 224, 224))
            
        # Get labels (-1 if cell not valid/too sparse)
        c_label = self.c_to_idx.get(data['coarse'], -1)
        m_label = self.m_to_idx.get(data['medium'], -1) if data['medium'] else -1
        f_label = self.f_to_idx.get(data['fine'], -1) if data['fine'] else -1
        
        return image, c_label, m_label, f_label

# --- 2. Model (Same Architecture) ---
class PIGEONModel(nn.Module):
    def __init__(self, clip_model, n_c, n_m, n_f):
        super().__init__()
        self.clip = clip_model
        # Freeze CLIP
        for p in self.clip.parameters():
            p.requires_grad = False
            
        embed_dim = 512
        
        self.head_c = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_c)
        )
        self.head_m = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_m)
        )
        self.head_f = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_f)
        )
        
    def forward(self, x):
        with torch.no_grad():
            feat = self.clip.encode_image(x).float()
        return self.head_c(feat), self.head_m(feat), self.head_f(feat)

# --- 3. Training Loop ---
def train():
    # Load CLIP
    clip_model, preprocess = clip.load("ViT-B/32", device=DEVICE)
    
    # Dataset
    ds = ExpandedGeolocDataset(transform=preprocess)
    
    # 80/20 Split
    train_size = int(0.8 * len(ds))
    val_size = len(ds) - train_size
    train_ds, val_ds = torch.utils.data.random_split(ds, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=2)
    
    # Init Model
    n_c = len(ds.c_to_idx)
    n_m = len(ds.m_to_idx)
    n_f = len(ds.f_to_idx)
    print(f"Classes: {n_c} Coarse, {n_m} Medium, {n_f} Fine")
    
    model = PIGEONModel(clip_model, n_c, n_m, n_f).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    
    best_loss = float('inf')
    
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        
        # Train
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for imgs, c, m, f in pbar:
            imgs, c, m, f = imgs.to(DEVICE), c.to(DEVICE), m.to(DEVICE), f.to(DEVICE)
            
            optimizer.zero_grad()
            out_c, out_m, out_f = model(imgs)
            
            # Hierarchical Loss (Weighted)
            loss = criterion(out_c, c) + 0.5 * criterion(out_m, m) + 0.25 * criterion(out_f, f)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
            
        # Validate
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for imgs, c, m, f in val_loader:
                imgs, c, m, f = imgs.to(DEVICE), c.to(DEVICE), m.to(DEVICE), f.to(DEVICE)
                out_c, out_m, out_f = model(imgs)
                
                loss = criterion(out_c, c) + 0.5 * criterion(out_m, m) + 0.25 * criterion(out_f, f)
                val_loss += loss.item()
                
                # Simple accuracy on coarse level
                pred = out_c.argmax(dim=1)
                mask = c != -1
                if mask.sum() > 0:
                    correct += (pred[mask] == c[mask]).sum().item()
                    total += mask.sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        acc = 100 * correct / total if total > 0 else 0
        print(f"  Val Loss: {avg_val_loss:.4f} | Coarse Acc: {acc:.2f}%")
        
        # Save Best
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), "pigeon_expanded_best.pth")
            print("  ✓ Saved Best Model")

if __name__ == "__main__":
    train()

🚀 STARTING EXPANDED TRAINING on cuda
Classes: 107 Coarse, 150 Medium, 135 Fine


Epoch 1/20: 100%|██████████| 74/74 [00:09<00:00,  8.05it/s, loss=5.84]


  Val Loss: 6.6584 | Coarse Acc: 29.35%
  ✓ Saved Best Model


Epoch 2/20: 100%|██████████| 74/74 [00:09<00:00,  8.02it/s, loss=nan]


  Val Loss: 5.8548 | Coarse Acc: 34.81%
  ✓ Saved Best Model


...

  Val Loss: 5.5567 | Coarse Acc: 41.98%


Epoch 19/20: 100%|██████████| 74/74 [00:09<00:00,  8.06it/s, loss=nan]


  Val Loss: 5.6835 | Coarse Acc: 39.25%


Epoch 20/20: 100%|██████████| 74/74 [00:08<00:00,  8.26it/s, loss=0.0532]


  Val Loss: 5.6943 | Coarse Acc: 40.96%


In [24]:
"""
PIGEON Evaluation & Uncertainty Script
1. Calculates Distance Error (km)
2. Implements Uncertainty Quantification (Confidence Scores)
3. Generates "Localizability" metrics (Accuracy vs Confidence)
"""

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from PIL import Image
import clip
import json
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from s2sphere import CellId, LatLng

# --- CONFIG ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "pigeon_expanded_best.pth"

# --- HELPER FUNCTIONS ---
def get_lat_lon(cell_token):
    try:
        cell = CellId.from_token(cell_token)
        ll = cell.to_lat_lng()
        return ll.lat().degrees, ll.lng().degrees
    except:
        return 0.0, 0.0

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

# --- MODEL CLASS (Must match training) ---
class PIGEONModel(nn.Module):
    def __init__(self, clip_model, n_c, n_m, n_f):
        super().__init__()
        self.clip = clip_model
        embed_dim = 512
        self.head_c = nn.Sequential(nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_c))
        self.head_m = nn.Sequential(nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_m))
        self.head_f = nn.Sequential(nn.Linear(embed_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_f))
        
    def forward(self, x):
        with torch.no_grad():
            feat = self.clip.encode_image(x).float()
        return self.head_c(feat), self.head_m(feat), self.head_f(feat)

# --- MAIN EVALUATION ---
def evaluate():
    print("="*70)
    print("🚀 EVALUATING EXPANDED MODEL + UNCERTAINTY")
    print("="*70)
    
    # 1. Load Data Structures
    print("Loading data structures...")
    base_path = "data"
    with open(os.path.join(base_path, 'geocells.pkl'), 'rb') as f:
        geocells = pickle.load(f)
    with open(os.path.join(base_path, 'image_to_geocell.json'), 'r') as f:
        mapping = json.load(f)
        
    # Filter for TEST set only
    test_ids = [k for k, v in mapping.items() if v['split'] == 'test']
    print(f"Test Set Size: {len(test_ids)} images")

    # 2. Load Model
    print("Loading model...")
    clip_model, preprocess = clip.load("ViT-B/32", device=DEVICE)
    
    n_c, n_m, n_f = len(geocells['coarse']), len(geocells['medium']), len(geocells['fine'])
    model = PIGEONModel(clip_model, n_c, n_m, n_f).to(DEVICE)
    
    # Load weights
    state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    print("✓ Model loaded successfully")

    # 3. Run Inference
    results = []
    
    print("Running inference...")
    for img_id in tqdm(test_ids):
        # Load Image
        img_path = os.path.join(base_path, 'test_images', img_id)
        try:
            image = Image.open(img_path).convert('RGB')
            input_tensor = preprocess(image).unsqueeze(0).to(DEVICE)
        except:
            continue

        # Predict
        with torch.no_grad():
            logits_c, logits_m, logits_f = model(input_tensor)
            
            # --- UNCERTAINTY QUANTIFICATION ---
            # Calculate Confidence (Softmax Probability)
            probs = torch.softmax(logits_c, dim=1)
            confidence, pred_idx = probs.max(dim=1)
            confidence = confidence.item()
            pred_idx = pred_idx.item()
            
            # Get Prediction Coords
            pred_token = geocells['coarse'][pred_idx]
            pred_lat, pred_lon = get_lat_lon(pred_token)
            
            # Get True Coords
            true_lat = mapping[img_id]['lat']
            true_lon = mapping[img_id]['lon']
            
            # Calculate Error
            error_km = haversine(true_lat, true_lon, pred_lat, pred_lon)
            
            results.append({
                'img_id': img_id,
                'true_lat': true_lat, 'true_lon': true_lon,
                'pred_lat': pred_lat, 'pred_lon': pred_lon,
                'error_km': error_km,
                'confidence': confidence  # <--- THIS IS YOUR PROPOSAL METRIC
            })

    # 4. Analysis & Metrics
    df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print("📊 FINAL RESULTS")
    print("="*70)
    print(f"Median Error: {df['error_km'].median():.2f} km")
    print(f"Mean Error:   {df['error_km'].mean():.2f} km")
    print(f"Accuracy @ 25km:  {(df['error_km'] < 25).mean()*100:.1f}%")
    print(f"Accuracy @ 200km: {(df['error_km'] < 200).mean()*100:.1f}%")
    print(f"Accuracy @ 750km: {(df['error_km'] < 750).mean()*100:.1f}%")
    
    # --- LOCALIZABILITY (Proposal Item #3) ---
    print("\n📈 UNCERTAINTY & LOCALIZABILITY")
    print("-" * 30)
    print("Does the model know when it's wrong? (Filtering by confidence)")
    
    thresholds = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]
    print(f"{'Conf >':<10} | {'Retained':<10} | {'Median Err':<12} | {'Acc @ 750km'}")
    
    for t in thresholds:
        subset = df[df['confidence'] > t]
        if len(subset) == 0:
            continue
        retained_pct = (len(subset) / len(df)) * 100
        median_err = subset['error_km'].median()
        acc_750 = (subset['error_km'] < 750).mean() * 100
        print(f"{t:<10} | {retained_pct:>5.1f}%     | {median_err:>8.2f} km | {acc_750:>8.1f}%")

    # Save
    df.to_csv("expanded_results_with_uncertainty.csv", index=False)
    print("\n✓ Results saved to expanded_results_with_uncertainty.csv")
    print("="*70)

if __name__ == "__main__":
    evaluate()

🚀 EVALUATING EXPANDED MODEL + UNCERTAINTY
Loading data structures...
Test Set Size: 195 images
Loading model...
✓ Model loaded successfully
Running inference...


100%|██████████| 195/195 [00:04<00:00, 43.55it/s]


📊 FINAL RESULTS
Median Error: 238.85 km
Mean Error:   1126.14 km
Accuracy @ 25km:  0.5%
Accuracy @ 200km: 36.4%
Accuracy @ 750km: 80.5%

📈 UNCERTAINTY & LOCALIZABILITY
------------------------------
Does the model know when it's wrong? (Filtering by confidence)
Conf >     | Retained   | Median Err   | Acc @ 750km
0.0        | 100.0%     |   238.85 km |     80.5%
0.1        | 100.0%     |   238.85 km |     80.5%
0.3        |  76.9%     |   232.46 km |     89.3%
0.5        |  51.3%     |   204.28 km |     95.0%
0.7        |  28.7%     |   196.95 km |     94.6%
0.9        |   9.2%     |   160.65 km |     94.4%

✓ Results saved to expanded_results_with_uncertainty.csv
